# BS5: ONE Dataset -> NEW training -> best checkpoints -> continuous eval600
Upload the NEW source ZIP as one Dataset, attach it, enable GPU, then Run All.
No old audit/weights required. NEW C-R/C-N x seeds551/552, 60k/run (240k total).
Frozen-best evaluation: horizon600, controls1200 + main24000. One final download includes best weights.


In [ ]:
import os, sys, json, pathlib, subprocess, hashlib
os.environ['PYTHONDONTWRITEBYTECODE']='1'
INPUT=pathlib.Path('/kaggle/input'); WORK=pathlib.Path('/kaggle/working')
sources=sorted({p.parents[2] for p in INPUT.rglob('run_bs5_retrain_eval600.py') if p.parent.name=='evaluation'})
assert len(sources)==1, f'Attach ONE new source Dataset: {sources}'
SRC=sources[0]; os.chdir(SRC)
OUT=WORK/'bs5_retrain_eval600'
def command(*args):
    subprocess.run([sys.executable,'-B','-m','scripts.evaluation.run_bs5_retrain_eval600',*map(str,args)],cwd=SRC,check=True)
print('Source:',SRC,'Output:',OUT)
print('NEW training240k; separate eval cap25200; no historical checkpoint required.')


In [ ]:
import torch, numpy as np
print('Runtime:',sys.version,torch.__version__,np.__version__,'CUDA:',torch.cuda.is_available())
assert torch.cuda.is_available(), 'Enable GPU in Kaggle Notebook Settings.'
if not (OUT/'workflow.json').is_file():
    assert not OUT.exists(), 'Keep partial preparation evidence; do not overwrite it.'
    command('prepare','--output',OUT,'--device','cuda')
else:
    print('Existing experiment: preserve it. Cell3 continues committed training; no new lineage.')


In [ ]:
# Existing training runner: NEW prefix24k -> OWN tail36k, then best validation150.
# Best weights are frozen before continuous600. No training updates during evaluation.
# Re-running skips completed work. An interrupted eval game cannot be reset/retried.
command('run','--output',OUT)


In [ ]:
from IPython.display import FileLink, display
receipt=OUT/'delivery_receipt.json'
assert receipt.is_file(), 'Not complete. Preserve output, inspect the previous error; do not restart a new budget.'
info=json.loads(receipt.read_text()); package=pathlib.Path(info['path'])
assert info['accepted'] and info['includes_selected_weights']
h=hashlib.sha256()
with package.open('rb') as stream:
    for chunk in iter(lambda:stream.read(1024*1024),b''): h.update(chunk)
assert h.hexdigest()==info['sha256']
print('Download this ONE results ZIP. It includes training audit + repro with best weights + eval600 evidence.')
display(FileLink(str(package))); display(FileLink(str(receipt)))
